In [4]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score

In [2]:
df = pd.read_csv("/Users/theerdhasjoy/BIA/ML/Dataset/Housing.csv")
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [7]:
X = df.drop('price', axis= 1)
y = df['price']

In [14]:
num_cols = X.select_dtypes(include = ["int64","int32","float32","float64"]).columns.tolist()
cat_cols = X.select_dtypes(include= ["object"]).columns.tolist()

/var/folders/y5/lf_stmcj5nv2hnzy3q1b5pb80000gn/T/ipykernel_10583/2956993741.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include= ["object"]).columns.tolist()


In [15]:
num_fill = X[num_cols].median()
cat_fill = X[cat_cols].mode().iloc[0]
 
# Fill missing values
 
X[num_cols] = X[num_cols].fillna(num_fill)
X[cat_cols] = X[cat_cols].fillna(cat_fill)

In [16]:
X["furnishingstatus"] = X["furnishingstatus"].map({
    "unfurnished": 0,
    "semi-furnished": 1,
    "furnished": 2
})
 
X['guestroom'] = X['guestroom'].map({"yes":1,"no":0})

In [17]:
# Custom encoders
encoders = {
    "mainroad": {"yes": 1, "no": 0},
    "guestroom": {"yes": 1, "no": 0},
    "basement": {"yes": 1, "no": 0},
    "hotwaterheating": {"yes": 1, "no": 0},
    "airconditioning": {"yes": 1, "no": 0},
    "prefarea": {"yes": 1, "no": 0},
    "furnishingstatus": {
        "unfurnished": 0,
        "semi-furnished": 1,
        "furnished": 2
    }
}
 
# Apply encoding
for col, mapping in encoders.items():
    X[col] = X[col].map(mapping)
 
from sklearn.model_selection import train_test_split
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
 
from sklearn.ensemble import RandomForestRegressor
 
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
 
 
import joblib
 
preprocessing = {
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "num_fill": num_fill,
    "cat_fill": cat_fill,
    "encoders": encoders,
    "columns": X.columns.tolist()
}
 
joblib.dump(preprocessing, "preprocessing.joblib")
joblib.dump(model, "model.joblib")
 
pred=model.predict(X_test)
print('MAE:',mean_absolute_error(y_test,pred))
print('R2 :',r2_score(y_test,pred))

MAE: 1068837.2044342507
R2 : 0.5828981020993677


In [18]:
import joblib
import pandas as pd
 
# Load model and preprocessing
model = joblib.load("model.joblib")
preprocessing = joblib.load("preprocessing.joblib")
 
# New house details
new_house = {
    "area": 7420,
    "bedrooms": 4,
    "bathrooms": 2,
    "stories": 3,
    "mainroad": "yes",
    "guestroom": "no",
    "basement": "no",
    "hotwaterheating": "no",
    "airconditioning": "yes",
    "parking": 2,
    "prefarea": "yes",
    "furnishingstatus": "furnished"
}
 
# Convert to DataFrame
input_df = pd.DataFrame([new_house])
 


 
# Fill missing values
input_df[preprocessing["num_cols"]] = input_df[
    preprocessing["num_cols"]
].fillna(preprocessing["num_fill"])
 
input_df[preprocessing["cat_cols"]] = input_df[
    preprocessing["cat_cols"]
].fillna(preprocessing["cat_fill"])
 
# Apply custom encoding
for col, mapping in preprocessing["encoders"].items():
    input_df[col] = input_df[col].map(mapping)
 
# Arrange columns in the same order as training
input_df = input_df[preprocessing["columns"]]
 
# Predict
prediction = model.predict(input_df)
 
print(f"Predicted House Price: ₹{prediction[0]:,.2f}")
 

Predicted House Price: ₹8,725,150.00
